In [1]:
import os

os.chdir('..')

## Loading the Dataset:

In this section the pointcloud is loaded. The SIREN paper suggests normalizing the point coordinates as periodic activations implicitly expect a bounded input. 

In [2]:
import open3d as o3d
import numpy as np
import torch
import matplotlib.pyplot as plt
import src.model.SIREN as si
from src.model.training import train
import src.loss.SDF_loss as loss
from src.mesh_extraction.marching_cubes_test import write_obj
import src.model.MLP as simple
import src.data.dataset as data

mesh = data.MeshDataset('data/pointclouds/Armadillo/Armadillo.ply')
print(mesh.vertices)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[[ 0.07790768 -0.12729204  0.36060192]
 [-0.7047902   0.60381086 -0.75930651]
 [ 0.04956958 -0.06494814  0.3892928 ]
 ...
 [-0.00709846 -0.36533849  0.56706102]
 [-0.65767778 -0.97143663  0.01645928]
 [-0.76626174  0.54733239 -0.66349749]]


## Defining the Model

In this cell we will define the SIREN model. This particular INR uses sine activations for nonlinearity and is supposed to capture more information given the underlying data when compared to a model that uses ReLU activations. This way, a good INR accuracy can be achieved with fewer neurons.

In [3]:
size_per_layer = 256
model = si.SIRENSDF()
model_loss = loss.Loss(lambda_surface=150,  lambda_eikonal=0.5, lambda_normal=1.5, lambda_inter=0.5, lambda_sign=1.5, lambda_twd=1e-4, k=int(size_per_layer/5), model=model) # optional normal loss if normals contained in the pointcloud
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


## Model training



In [4]:
# TODO: introduce other pruning methods and think about implementing a ReLU based model
train(epochs=500, data=mesh, no_surface=10000, no_off_surface=15000, model=model, loss=model_loss, optimizer=optimizer, prune=False, scene=mesh.scene)

NameError: name 'normal_constraint' is not defined

#### Model size after pruning

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Sample a few surface points and check their SDF values
test_points = mesh.vertices[:10]  # First 10 points
test_tensor = torch.from_numpy(test_points).float().to(device)
with torch.no_grad():
    sdf_values = model(test_tensor)
print("SDF values for surface points:")
print(sdf_values)
print("Mean absolute SDF:", torch.abs(sdf_values).mean().item())

SDF values for surface points:
tensor([[-0.0168],
        [-0.0169],
        [-0.0171],
        [-0.0170],
        [-0.0165],
        [-0.0170],
        [-0.0168],
        [-0.0167],
        [-0.0168],
        [-0.0169]], device='cuda:0')
Mean absolute SDF: 0.016847610473632812


In [ ]:
import src.mesh_extraction.marching_cubes_gpu as marching_cubes
marching_cubes.write_obj("lucy_pred_128_siren.obj", model=model, resolution=128, level=0.0)


In [ ]:
# import src.mesh_extraction.marching_cubes_test as marching_cubes_test
# marching_cubes_test.write_obj("armadillo_mesh_ground_truth_128.obj", scene=mesh.scene, resolution=128, level=0.0)


In [ ]:
# Make batched point
test_point = torch.from_numpy(np.array([[-1, -1, -1]])).float().to(device)

# Compute SIREN model prediction
sdf_pred = model(test_point)
print("SIREN prediction:", sdf_pred)

# Compute Open3D signed distance
distance = mesh.scene.compute_signed_distance(
    o3d.core.Tensor(test_point.cpu().numpy(), dtype=o3d.core.Dtype.Float32)
)
print("Open3D SDF:", distance.numpy())

SIREN prediction: tensor([[0.1795]], device='cuda:0', grad_fn=<AddmmBackward0>)
Open3D SDF: [1.1372143]
